https://www.kaggle.com/datasets/devchauhan1/market-basket-optimisationcsv?resource=download


# Apriori Algorithm

## Overview

The **Apriori algorithm** is a classic algorithm used in **Association Rule Learning** to discover frequent itemsets and generate association rules from transactional datasets.

It is commonly used in:

- Market Basket Analysis
- Recommendation Systems
- Pattern Discovery

---

## Core Idea (Apriori Principle)

> If an itemset is frequent, then all of its subsets must also be frequent.

This property significantly reduces the search space by pruning non-frequent itemsets early.

---

## How Apriori Works (Steps)

1. **Set Minimum Support**
   - Define a threshold to determine which itemsets are considered frequent.

2. **Generate Frequent 1-Itemsets**
   - Calculate support for each individual item.
   - Remove items below `min_support`.

3. **Generate Candidate k-Itemsets**
   - Combine frequent (k-1)-itemsets to form k-itemsets.

4. **Prune Non-Frequent Itemsets**
   - Remove itemsets whose support < `min_support`.

5. **Repeat**
   - Continue until no new frequent itemsets are found.

6. **Generate Association Rules**
   - From frequent itemsets, generate rules using confidence and lift.

---

## Important Metrics

### 1. Support

Measures how frequently an itemset appears in the dataset.

$$
Support(A) = \frac{\text{Number of transactions containing A}}{\text{Total transactions}}
$$

---

### 2. Confidence

Measures how often B occurs given that A has occurred.

$$
Confidence(A \rightarrow B) = \frac{Support(A \cap B)}{Support(A)}
$$

---

### 3. Lift

Measures how much more likely B is to occur with A compared to random chance.

$$
Lift(A \rightarrow B) = \frac{Confidence(A \rightarrow B)}{Support(B)}
$$

- Lift > 1 → Positive association
- Lift = 1 → No association
- Lift < 1 → Negative association

---

## Key Parameters in Apriori (mlxtend)

### `min_support`

- Minimum support threshold.
- Filters out infrequent itemsets.
- Example: `min_support=0.3`

### `max_len`

- Maximum length of itemsets.
- Example: `max_len=3`

---

## Parameters in Association Rules

### `metric`

- Metric used to filter rules.
- Options: `"confidence"`, `"lift"`, `"support"`

---

### `min_threshold`

- Minimum value for the selected metric.
- Example: `min_threshold=0.7`

---

## Advantages

- Easy to understand and implement
- Strong theoretical foundation
- Works well on small to medium datasets

---

## Limitations

- Computationally expensive for large datasets
- Generates many candidate itemsets
- Not suitable for high-dimensional big data
- Go through the dataset a lot to calculate the **support**.

---

## Faster Alternative

For large datasets, consider using **FP-Growth**, which avoids candidate generation and is more efficient.


In [4]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')


In [5]:
dataset = pd.read_csv('Market_Basket_Optimisation.csv', header=None)
dataset.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,shrimp,almonds,avocado,vegetables mix,green grapes,whole weat flour,yams,cottage cheese,energy drink,tomato juice,low fat yogurt,green tea,honey,salad,mineral water,salmon,antioxydant juice,frozen smoothie,spinach,olive oil
1,burgers,meatballs,eggs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,chutney,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,turkey,avocado,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,mineral water,milk,energy bar,whole wheat rice,green tea,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7501 entries, 0 to 7500
Data columns (total 20 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       7501 non-null   object
 1   1       5747 non-null   object
 2   2       4389 non-null   object
 3   3       3345 non-null   object
 4   4       2529 non-null   object
 5   5       1864 non-null   object
 6   6       1369 non-null   object
 7   7       981 non-null    object
 8   8       654 non-null    object
 9   9       395 non-null    object
 10  10      256 non-null    object
 11  11      154 non-null    object
 12  12      87 non-null     object
 13  13      47 non-null     object
 14  14      25 non-null     object
 15  15      8 non-null      object
 16  16      4 non-null      object
 17  17      4 non-null      object
 18  18      3 non-null      object
 19  19      1 non-null      object
dtypes: object(20)
memory usage: 1.1+ MB


In [7]:
dataset.values

array([['shrimp', 'almonds', 'avocado', ..., 'frozen smoothie',
        'spinach', 'olive oil'],
       ['burgers', 'meatballs', 'eggs', ..., nan, nan, nan],
       ['chutney', nan, nan, ..., nan, nan, nan],
       ...,
       ['chicken', nan, nan, ..., nan, nan, nan],
       ['escalope', 'green tea', nan, ..., nan, nan, nan],
       ['eggs', 'frozen smoothie', 'yogurt cake', ..., nan, nan, nan]],
      shape=(7501, 20), dtype=object)

preprocessing


In [8]:
transactions = []
for i in range(0, 7501):
    transactions.append([str(dataset.values[i, j])  for j in range(0, 20) if pd.notna(dataset.values[i, j])])

transactions


[['shrimp',
  'almonds',
  'avocado',
  'vegetables mix',
  'green grapes',
  'whole weat flour',
  'yams',
  'cottage cheese',
  'energy drink',
  'tomato juice',
  'low fat yogurt',
  'green tea',
  'honey',
  'salad',
  'mineral water',
  'salmon',
  'antioxydant juice',
  'frozen smoothie',
  'spinach',
  'olive oil'],
 ['burgers', 'meatballs', 'eggs'],
 ['chutney'],
 ['turkey', 'avocado'],
 ['mineral water', 'milk', 'energy bar', 'whole wheat rice', 'green tea'],
 ['low fat yogurt'],
 ['whole wheat pasta', 'french fries'],
 ['soup', 'light cream', 'shallot'],
 ['frozen vegetables', 'spaghetti', 'green tea'],
 ['french fries'],
 ['eggs', 'pet food'],
 ['cookies'],
 ['turkey', 'burgers', 'mineral water', 'eggs', 'cooking oil'],
 ['spaghetti', 'champagne', 'cookies'],
 ['mineral water', 'salmon'],
 ['mineral water'],
 ['shrimp',
  'chocolate',
  'chicken',
  'honey',
  'oil',
  'cooking oil',
  'low fat yogurt'],
 ['turkey', 'eggs'],
 ['turkey',
  'fresh tuna',
  'tomatoes',
  'spagh

In [9]:
# pip install apyori

In [10]:
from apyori import apriori

rules = apriori(transactions, min_support=0.003, min_confidence=0.2, min_lift=3, min_length=2)
rules


<generator object apriori at 0x000001FA0E1BB060>

In [11]:
rules_list = list(rules)
rules_list

[RelationRecord(items=frozenset({'chicken', 'light cream'}), support=0.004532728969470737, ordered_statistics=[OrderedStatistic(items_base=frozenset({'light cream'}), items_add=frozenset({'chicken'}), confidence=0.29059829059829057, lift=4.84395061728395)]),
 RelationRecord(items=frozenset({'escalope', 'mushroom cream sauce'}), support=0.005732568990801226, ordered_statistics=[OrderedStatistic(items_base=frozenset({'mushroom cream sauce'}), items_add=frozenset({'escalope'}), confidence=0.3006993006993007, lift=3.790832696715049)]),
 RelationRecord(items=frozenset({'escalope', 'pasta'}), support=0.005865884548726837, ordered_statistics=[OrderedStatistic(items_base=frozenset({'pasta'}), items_add=frozenset({'escalope'}), confidence=0.3728813559322034, lift=4.700811850163794)]),
 RelationRecord(items=frozenset({'honey', 'fromage blanc'}), support=0.003332888948140248, ordered_statistics=[OrderedStatistic(items_base=frozenset({'fromage blanc'}), items_add=frozenset({'honey'}), confidence=0

In [12]:
len(rules_list)

80

In [13]:
rules_list[0]

RelationRecord(items=frozenset({'chicken', 'light cream'}), support=0.004532728969470737, ordered_statistics=[OrderedStatistic(items_base=frozenset({'light cream'}), items_add=frozenset({'chicken'}), confidence=0.29059829059829057, lift=4.84395061728395)])

In [14]:
# 0 index is the first rule, 0 index of the first rule is the first itemset of the first rule
rules_list[0][0]

frozenset({'chicken', 'light cream'})

In [15]:
# 0 index is the first rule, 2 index of the first rule is the lift of the first rule
rules_list[0][2]

[OrderedStatistic(items_base=frozenset({'light cream'}), items_add=frozenset({'chicken'}), confidence=0.29059829059829057, lift=4.84395061728395)]

In [16]:
rules_list[0][2][0][3]

4.84395061728395

In [17]:
def inspect(rules):
    lhs = [tuple(rule[0])[0] for rule in rules]
    rhs = [tuple(rule[0])[1] for rule in rules]
    supports = [rule[1] for rule in rules]
    confidences = [rule[2][0][2] for rule in rules]
    lifts = [rule[2][0][3] for rule in rules]
    return list(zip(lhs, rhs, supports, confidences, lifts))
resultsinDataFrame = pd.DataFrame(inspect(rules_list), columns=['Left Hand Side', 'Right Hand Side', 'Support', 'Confidence', 'Lift'])
resultsinDataFrame

,Left Hand Side,Right Hand Side,Support,Confidence,Lift
0,chicken,light cream,0.004533,0.290598,4.843951
1,escalope,mushroom cream sauce,0.005733,0.300699,3.790833
2,escalope,pasta,0.005866,0.372881,4.700812
3,honey,fromage blanc,0.003333,0.245098,5.164271
4,ground beef,herb & pepper,0.015998,0.323450,3.291994
...,...,...,...,...,...
75,spaghetti,ground beef,0.003066,0.216981,3.632981
76,pancakes,ground beef,0.003066,0.211009,3.532991
77,tomatoes,spaghetti,0.003066,0.261364,4.376091
78,milk,spaghetti,0.003333,0.211864,3.216994


In [18]:
resultsinDataFrame.sort_values(by='Lift', ascending=False)

,Left Hand Side,Right Hand Side,Support,Confidence,Lift
70,soup,milk,0.003066,0.383333,7.987176
69,milk,frozen vegetables,0.003333,0.294118,6.128268
52,whole wheat pasta,mineral water,0.003866,0.402778,6.115863
44,spaghetti,ground beef,0.003066,0.216981,5.535971
3,honey,fromage blanc,0.003333,0.245098,5.164271
...,...,...,...,...,...
23,spaghetti,red wine,0.003733,0.528302,3.034297
36,tomatoes,green tea,0.003066,0.207207,3.029749
71,milk,frozen vegetables,0.004533,0.288136,3.022804
32,shrimp,frozen vegetables,0.005999,0.215311,3.013149


# FP-Growth Algorithm (Frequent Pattern Growth)

## Overview

FP-Growth (Frequent Pattern Growth) is an efficient algorithm used to mine frequent itemsets without generating candidate itemsets explicitly (unlike Apriori).

It uses a special tree structure called an **FP-Tree (Frequent Pattern Tree)** to store compressed transactional data.

FP-Growth is especially useful for **large datasets** where Apriori becomes computationally expensive.

---

## Why FP-Growth?

Apriori suffers from:

- Generating too many candidate itemsets
- Multiple database scans
- High computational cost for large datasets

FP-Growth solves this by:

- Compressing the dataset into a tree
- Avoiding candidate generation
- Reducing database scans

---

# Steps of FP-Growth Algorithm

## Step 1: Scan the Dataset and Compute Support

- Count the frequency of each item.
- Remove items below `min_support`.
- Sort remaining items in descending order of support.

This creates the **Frequent Item List**.

---

## Step 2: Construct the FP-Tree

For each transaction:

1. Keep only frequent items.
2. Sort items according to global frequency order.
3. Insert them into the FP-Tree.

### FP-Tree Structure:

- Root node (null)
- Branches representing transactions
- Shared prefixes are merged (data compression)
- Each node contains:
  - Item name
  - Count
  - Link to similar items (node-link structure)

---

## Step 3: Mine Frequent Patterns from FP-Tree

The algorithm works recursively:

1. Start from the least frequent item.
2. Extract its conditional pattern base.
3. Construct a conditional FP-Tree.
4. Generate frequent itemsets.
5. Repeat recursively.

This process is called **Divide-and-Conquer Strategy**.

---

# Key Parameters

| Parameter   | Description                                             |
| ----------- | ------------------------------------------------------- |
| min_support | Minimum support threshold to consider itemsets frequent |

FP-Growth does not require candidate generation parameter tuning like Apriori.

---

# Advantages of FP-Growth

- Faster than Apriori on large datasets
- No candidate generation
- Fewer database scans (only 2 scans)
- Efficient memory usage due to tree compression
- Works well with dense datasets

---

# Limitations

- Tree structure can be memory-intensive if data is highly diverse
- More complex to implement than Apriori
- Harder to parallelize in some cases

---

# FP-Growth vs Apriori

| Feature              | Apriori                | FP-Growth             |
| -------------------- | ---------------------- | --------------------- |
| Candidate Generation | Yes                    | No                    |
| Database Scans       | Multiple               | Only 2                |
| Speed                | Slower on large data   | Faster on large data  |
| Memory Usage         | High (many candidates) | Compressed using tree |
| Implementation       | Simple                 | More complex          |
| Best For             | Small datasets         | Large datasets        |

---

# When to Use Each?

### Use Apriori when:

- Dataset is small
- You want simplicity
- Teaching basic frequent pattern mining

### Use FP-Growth when:

- Dataset is large
- Performance matters
- Many frequent patterns exist

---

# Final Insight

Both algorithms solve the same problem:  
Finding frequent itemsets.

The main difference is in efficiency:

- Apriori uses **candidate generation + pruning**
- FP-Growth uses **tree compression + recursive mining**

For real-world large-scale market basket analysis, FP-Growth is usually preferred.


In [19]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

In [20]:
# One-Hot Encode using TransactionEncoder
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_array, columns=te.columns_)
df_encoded.head()

,asparagus,almonds,antioxydant juice,asparagus,avocado,babies food,bacon,barbecue sauce,black tea,blueberries,...,turkey,vegetables mix,water spray,white wine,whole weat flour,whole wheat pasta,whole wheat rice,yams,yogurt cake,zucchini
0,False,True,True,False,True,False,False,False,False,False,...,False,True,False,False,True,False,False,True,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,True,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False


In [21]:
df_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7501 entries, 0 to 7500
Columns: 120 entries,  asparagus to zucchini
dtypes: bool(120)
memory usage: 879.2 KB


In [22]:
#  Apply FP-Growth
frequent_itemsets = fpgrowth(df_encoded, min_support=0.01, use_colnames=True)

print("Frequent Itemsets (Top 20):")
frequent_itemsets.head(20)


Frequent Itemsets (Top 20):


,support,itemsets
0,0.238368,(mineral water)
1,0.132116,(green tea)
2,0.076523,(low fat yogurt)
3,0.071457,(shrimp)
4,0.065858,(olive oil)
5,0.063325,(frozen smoothie)
6,0.047460,(honey)
7,0.042528,(salmon)
8,0.033329,(avocado)
9,0.031862,(cottage cheese)


In [23]:
frequent_itemsets.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 257 entries, 0 to 256
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   support   257 non-null    float64
 1   itemsets  257 non-null    object 
dtypes: float64(1), object(1)
memory usage: 4.1+ KB


In [24]:
#  Generate association rules
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)

print("\nAssociation Rules (Top 20):")
rules.head(20)


Association Rules (Top 20):


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(spaghetti),(green tea),0.174110,0.132116,0.026530,0.152374,1.153335,1.0,0.003527,1.023900,0.160977,0.094852,0.023342,0.176590
1,(green tea),(spaghetti),0.132116,0.174110,0.026530,0.200807,1.153335,1.0,0.003527,1.033405,0.153188,0.094852,0.032325,0.176590
2,(green tea),(french fries),0.132116,0.170911,0.028530,0.215943,1.263488,1.0,0.005950,1.057436,0.240286,0.103934,0.054316,0.191435
3,(french fries),(green tea),0.170911,0.132116,0.028530,0.166927,1.263488,1.0,0.005950,1.041786,0.251529,0.103934,0.040110,0.191435
4,(green tea),(chocolate),0.132116,0.163845,0.023464,0.177598,1.083943,1.0,0.001817,1.016724,0.089231,0.086106,0.016449,0.160402
5,(chocolate),(green tea),0.163845,0.132116,0.023464,0.143206,1.083943,1.0,0.001817,1.012944,0.092617,0.086106,0.012778,0.160402
6,(green tea),(eggs),0.132116,0.179709,0.025463,0.192735,1.072479,1.0,0.001721,1.016135,0.077869,0.088920,0.015879,0.167213
7,(eggs),(green tea),0.179709,0.132116,0.025463,0.141691,1.072479,1.0,0.001721,1.011156,0.082387,0.088920,0.011033,0.167213
8,(mineral water),(low fat yogurt),0.238368,0.076523,0.023997,0.100671,1.315565,1.0,0.005756,1.026851,0.314943,0.082493,0.026149,0.207130
9,(low fat yogurt),(mineral water),0.076523,0.238368,0.023997,0.313589,1.315565,1.0,0.005756,1.109585,0.259747,0.082493,0.098762,0.207130


In [25]:
rules.iloc[: , 0:7].sort_values(by='lift', ascending=False).head(20)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
379,(herb & pepper),(ground beef),0.049460,0.098254,0.015998,0.323450,3.291994
378,(ground beef),(herb & pepper),0.098254,0.049460,0.015998,0.162822,3.291994
350,(ground beef),"(spaghetti, mineral water)",0.098254,0.059725,0.017064,0.173677,2.907928
347,"(spaghetti, mineral water)",(ground beef),0.059725,0.098254,0.017064,0.285714,2.907928
61,(olive oil),"(spaghetti, mineral water)",0.065858,0.059725,0.010265,0.155870,2.609786
56,"(spaghetti, mineral water)",(olive oil),0.059725,0.065858,0.010265,0.171875,2.609786
281,(frozen vegetables),(tomatoes),0.095321,0.068391,0.016131,0.169231,2.474464
280,(tomatoes),(frozen vegetables),0.068391,0.095321,0.016131,0.235867,2.474464
34,(shrimp),(frozen vegetables),0.071457,0.095321,0.016664,0.233209,2.446574
35,(frozen vegetables),(shrimp),0.095321,0.071457,0.016664,0.174825,2.446574


In [34]:
def get_closed_patterns(frequent_itemsets):
    closed_patterns = []

    for i, row_i in frequent_itemsets.iterrows():
        is_closed = True
        
        for j, row_j in frequent_itemsets.iterrows():
            if row_i['itemsets'] < row_j['itemsets']:  # proper subset
                if row_i['support'] <= row_j['support']:
                    is_closed = False
                    break
                    
        if is_closed:
            closed_patterns.append(row_i)

    return pd.DataFrame(closed_patterns)

closed_itemsets = get_closed_patterns(frequent_itemsets)
print("Closed Itemsets (Top 20):")
closed_itemsets.head(20)

Closed Itemsets (Top 20):


,support,itemsets
0,0.238368,(mineral water)
1,0.132116,(green tea)
2,0.076523,(low fat yogurt)
3,0.071457,(shrimp)
4,0.065858,(olive oil)
5,0.063325,(frozen smoothie)
6,0.047460,(honey)
7,0.042528,(salmon)
8,0.033329,(avocado)
9,0.031862,(cottage cheese)


In [30]:
def get_max_patterns(frequent_itemsets):
    max_patterns = []

    for i, row_i in frequent_itemsets.iterrows():
        is_max = True
        
        for j, row_j in frequent_itemsets.iterrows():
            if row_i['itemsets'] < row_j['itemsets']:  # proper subset
                is_max = False
                break
                
        if is_max:
            max_patterns.append(row_i)

    return pd.DataFrame(max_patterns)
max_itemsets = get_max_patterns(frequent_itemsets)
print("Maximal Itemsets (Top 20):")
max_itemsets.head(20)

Maximal Itemsets (Top 20):


,support,itemsets
9,0.031862,(cottage cheese)
10,0.030396,(tomato juice)
11,0.026663,(energy drink)
12,0.025730,(vegetables mix)
13,0.020397,(almonds)
14,0.011465,(yams)
17,0.020931,(meatballs)
21,0.027063,(energy bar)
23,0.029463,(whole wheat pasta)
25,0.015598,(light cream)


In [31]:
from mlxtend.frequent_patterns import fpgrowth, association_rules, fpmax
frequent_itemsets_fpmax = fpmax(df_encoded, min_support=0.01, use_colnames=True)
print("Closed Itemsets (Top 20):")
frequent_itemsets_fpmax.head(20)

Closed Itemsets (Top 20):


,support,itemsets
0,0.010399,(nonfat milk)
1,0.010532,(cider)
2,0.010799,(barbecue sauce)
3,0.010932,(magazines)
4,0.011465,(body spray)
5,0.011465,(yams)
6,0.011998,(extra dark chocolate)
7,0.011998,(melons)
8,0.013198,(eggplant)
9,0.013465,(gums)
